# Практикалық жұмыс 5 — MDP және мәндер итерациясы

Жеткізу роботы. Тор:

| | | | |
|---|---|---|---|
| A | B | C | **G (+10)** |
| D | # | E | **X (−10)** |
| **S** | F | H | I |

Ұяшықтарды ретімен іске қосыңыз: әр келесі ұяшық алдыңғыларда анықталған функцияларды қолданады.

## 1-тапсырма — MDP-ді анықтау

In [ ]:
# Күйлер, әрекеттер, бастапқы және терминалдық күйлер
states = ["A", "B", "C", "D", "E", "S", "F", "H", "I"]   # терминалдық емес күйлер
terminals = ["G", "X"]                                  # эпизод осы жерде аяқталады
actions = ["UP", "DOWN", "LEFT", "RIGHT"]
start = "S"

gamma = 0.9      # дисконттау коэффициенті
theta = 0.001    # тоқтату шарты: delta < theta

# Әр ұяшықтың орны (жол, баған). # қабырғасы бұл тізімде жоқ.
positions = {
    "A": (0, 0), "B": (0, 1), "C": (0, 2), "G": (0, 3),
    "D": (1, 0),              "E": (1, 2), "X": (1, 3),
    "S": (2, 0), "F": (2, 1), "H": (2, 2), "I": (2, 3),
}
state_at = {pos: s for s, pos in positions.items()}

print("Күйлер S:", states + terminals)
print("Әрекеттер A:", actions)
print("Бастапқы күй:", start)
print("Терминалдық күйлер:", terminals)
print("Марапаттар: G = +10, X = -10, қалыпты қозғалыс = -0.1")

Күйлер S: ['A', 'B', 'C', 'D', 'E', 'S', 'F', 'H', 'I', 'G', 'X']
Әрекеттер A: ['UP', 'DOWN', 'LEFT', 'RIGHT']
Бастапқы күй: S
Терминалдық күйлер: ['G', 'X']
Марапаттар: G = +10, X = -10, қалыпты қозғалыс = -0.1


## 2-тапсырма — Белгісіздікті қосу (ауысу ықтималдықтары)

0.8 — таңдалған бағыт, 0.1 — сол жаққа, 0.1 — оң жаққа ауытқу. Қабырғаға немесе тор шетіне тірелсе, робот орнында қалады.

In [ ]:
moves = {"UP": (-1, 0), "DOWN": (1, 0), "LEFT": (0, -1), "RIGHT": (0, 1)}

# әрекет: (негізгі бағыт, сол жаққа ауытқу, оң жаққа ауытқу)
deviations = {
    "UP":    ("UP", "LEFT", "RIGHT"),
    "DOWN":  ("DOWN", "RIGHT", "LEFT"),
    "LEFT":  ("LEFT", "DOWN", "UP"),
    "RIGHT": ("RIGHT", "UP", "DOWN"),
}


def move(state, direction):
    """Бір бағытта қозғалғаннан кейінгі күй. Қабырға/шекара -> орнында қалу."""
    row, col = positions[state]
    dr, dc = moves[direction]
    new_pos = (row + dr, col + dc)
    if new_pos not in state_at:
        return state
    return state_at[new_pos]


def reward(next_state, r_goal=10, r_danger=-10, r_step=-0.1):
    if next_state == "G":
        return r_goal
    if next_state == "X":
        return r_danger
    return r_step


def transitions(s, a, p_main=0.8, r_goal=10, r_danger=-10, r_step=-0.1):
    """[(ықтималдық, келесі_күй, марапат), ...] тізімін қайтарады."""
    p_side = round((1 - p_main) / 2, 10)
    result = []
    for direction, prob in zip(deviations[a], [p_main, p_side, p_side]):
        nxt = move(s, direction)
        result.append((prob, nxt, reward(nxt, r_goal, r_danger, r_step)))
    return result


def print_transition_table(s, a):
    print(f"{s} күйі, {a} әрекеті")
    print("Ағымдағы күй | Әрекет | Келесі күй | Ықтималдық | Марапат")
    total = 0
    for prob, nxt, r in transitions(s, a):
        print(f"{s:>12} | {a:>6} | {nxt:>10} | {prob:>10} | {r}")
        total += prob
    print("Тексеру: қосынды =", round(total, 2))
    print()


print_transition_table("H", "UP")      # нұсқаулықтағы мысал
print_transition_table("F", "RIGHT")
print_transition_table("E", "RIGHT")
print_transition_table("C", "RIGHT")

H күйі, UP әрекеті
Ағымдағы күй | Әрекет | Келесі күй | Ықтималдық | Марапат
           H |     UP |          E |        0.8 | -0.1
           H |     UP |          F |        0.1 | -0.1
           H |     UP |          I |        0.1 | -0.1
Тексеру: қосынды = 1.0

F күйі, RIGHT әрекеті
Ағымдағы күй | Әрекет | Келесі күй | Ықтималдық | Марапат
           F |  RIGHT |          H |        0.8 | -0.1
           F |  RIGHT |          F |        0.1 | -0.1
           F |  RIGHT |          F |        0.1 | -0.1
Тексеру: қосынды = 1.0

E күйі, RIGHT әрекеті
Ағымдағы күй | Әрекет | Келесі күй | Ықтималдық | Марапат
           E |  RIGHT |          X |        0.8 | -10
           E |  RIGHT |          C |        0.1 | -0.1
           E |  RIGHT |          H |        0.1 | -0.1
Тексеру: қосынды = 1.0

C күйі, RIGHT әрекеті
Ағымдағы күй | Әрекет | Келесі күй | Ықтималдық | Марапат
           C |  RIGHT |          G |        0.8 | 10
           C |  RIGHT |          C |        0.1 | -0.1
         

## 3-тапсырма — Марапаттар мен дисконттау

U = r₀ + γr₁ + γ²r₂ + γ³r₃ + …

In [ ]:
def discounted_utility(rewards, gamma):
    U = 0
    parts = []
    for t, r in enumerate(rewards):
        U += (gamma ** t) * r
        parts.append(f"{gamma}^{t}×{r}")
    print(f"{rewards}, γ={gamma}:  U = " + " + ".join(parts) + f" = {round(U, 4)}")
    return U


discounted_utility([1, 2, 3], 0.5)     # нұсқаулықтағы мысал: 2.75
discounted_utility([2, 4, 6], 0.5)
discounted_utility([2, 4, 6], 0.9)
discounted_utility([0, 0, 10], 0.5)

[1, 2, 3], γ=0.5:  U = 0.5^0×1 + 0.5^1×2 + 0.5^2×3 = 2.75
[2, 4, 6], γ=0.5:  U = 0.5^0×2 + 0.5^1×4 + 0.5^2×6 = 5.5
[2, 4, 6], γ=0.9:  U = 0.9^0×2 + 0.9^1×4 + 0.9^2×6 = 10.46
[0, 0, 10], γ=0.5:  U = 0.5^0×0 + 0.5^1×0 + 0.5^2×10 = 2.5


2.5

## 4-тапсырма — Q(s,a) және V(s) есептеу

Q(s,a) = Σ T(s,a,s′) [R(s,a,s′) + γV(s′)],  V(s) = maxₐ Q(s,a). Терминалдық күйлердің мәні 0.

In [ ]:
def q_value(s, a, V, gamma=0.9, **params):
    q = 0
    for prob, nxt, r in transitions(s, a, **params):
        next_value = 0 if nxt in terminals else V[nxt]
        q += prob * (r + gamma * next_value)
    return q


# Нұсқаулықтағы шешілген мысал: V(E)=5, V(F)=2, V(I)=1 -> Q(H,UP) = 3.77
V_example = {s: 0 for s in states}
V_example.update({"E": 5, "F": 2, "I": 1})
print("Q(H,UP) =", round(q_value("H", "UP", V_example), 4))

Q(H,UP) = 3.77


## 5-тапсырма — Мәндер итерациясын қолмен орындау (V₀ → V₁ → V₂)

In [ ]:
def bellman_update(V, gamma=0.9, **params):
    """Бір итерация: жаңа мәндер тек ЕСКІ V бойынша есептеледі."""
    new_V = {}
    for s in states:
        new_V[s] = max(q_value(s, a, V, gamma, **params) for a in actions)
    return new_V


V0 = {s: 0.0 for s in states}
V1 = bellman_update(V0)
V2 = bellman_update(V1)

print("Күй |   V0 |     V1 |     V2")
for s in ["C", "E", "H", "I"]:
    print(f"{s:>3} | {V0[s]:>4} | {V1[s]:>6.3f} | {V2[s]:>6.3f}")

print()
print("Q-мәндер (1-итерация, V0 бойынша):")
for s in ["C", "E", "H", "I"]:
    print(s, {a: round(q_value(s, a, V0), 3) for a in actions})

print()
print("Q-мәндер (2-итерация, V1 бойынша):")
for s in ["C", "E", "H", "I"]:
    print(s, {a: round(q_value(s, a, V1), 4) for a in actions})

Күй |   V0 |     V1 |     V2
  C |  0.0 |  7.980 |  8.689
  E |  0.0 | -0.100 |  4.647
  H |  0.0 | -0.100 | -0.190
  I |  0.0 | -0.100 | -0.190

Q-мәндер (1-итерация, V0 бойынша):
C {'UP': 0.91, 'DOWN': 0.91, 'LEFT': -0.1, 'RIGHT': 7.98}
E {'UP': -1.09, 'DOWN': -1.09, 'LEFT': -0.1, 'RIGHT': -8.02}
H {'UP': -0.1, 'DOWN': -0.1, 'LEFT': -0.1, 'RIGHT': -0.1}
I {'UP': -8.02, 'DOWN': -0.1, 'LEFT': -1.09, 'RIGHT': -1.09}

Q-мәндер (2-итерация, V1 бойынша):
C {'UP': 6.6466, 'DOWN': 0.829, 'LEFT': 0.5372, 'RIGHT': 8.6892}
E {'UP': 4.6466, 'DOWN': -1.171, 'LEFT': 0.5372, 'RIGHT': -7.3108}
H {'UP': -0.19, 'DOWN': -0.19, 'LEFT': -0.19, 'RIGHT': -0.19}
I {'UP': -8.038, 'DOWN': -0.19, 'LEFT': -1.171, 'RIGHT': -1.171}


## 6-тапсырма — Python-да мәндер итерациясы (толық цикл)

In [ ]:
def value_iteration(gamma=0.9, theta=0.001, verbose=True, **params):
    V = {s: 0.0 for s in states}      # V0(s) = 0
    iteration = 0
    while True:
        iteration += 1
        new_V = bellman_update(V, gamma, **params)
        delta = max(abs(new_V[s] - V[s]) for s in states)
        V = new_V
        if verbose:
            print(f"Итерация {iteration:2d}: delta = {delta:.6f}")
        if delta < theta:
            break
    return V, iteration


V, n_iter = value_iteration()
print("Жинақталды:", n_iter, "итерация")
print()
print("КҮЙЛЕРДІҢ СОҢҒЫ МӘНДЕРІ")
for s in states:
    print(f"V({s}) = {V[s]:.3f}")

Итерация  1: delta = 7.980000
Итерация  2: delta = 5.727600
Итерация  3: delta = 4.107672
Итерация  4: delta = 2.942944
Итерация  5: delta = 2.331454
Итерация  6: delta = 1.436748
Итерация  7: delta = 0.834876
Итерация  8: delta = 0.408040
Итерация  9: delta = 0.192484
Итерация 10: delta = 0.086133
Итерация 11: delta = 0.037804
Итерация 12: delta = 0.016206
Итерация 13: delta = 0.006871
Итерация 14: delta = 0.002877
Итерация 15: delta = 0.001197
Итерация 16: delta = 0.000494
Жинақталды: 16 итерация

КҮЙЛЕРДІҢ СОҢҒЫ МӘНДЕРІ
V(A) = 6.901
V(B) = 8.119
V(C) = 9.385
V(D) = 5.937
V(E) = 6.228
V(S) = 5.011
V(F) = 4.285
V(H) = 5.019
V(I) = 2.773


### Әр күйдің Q-мәндері, ең жақсы әрекет және соңғы саясат

In [ ]:
arrows = {"UP": "↑", "DOWN": "↓", "LEFT": "←", "RIGHT": "→"}


def get_policy(V, gamma=0.9, **params):
    policy = {}
    for s in states:
        q = {a: q_value(s, a, V, gamma, **params) for a in actions}
        policy[s] = max(q, key=q.get)
    return policy


def print_grid(policy):
    layout = [["A", "B", "C", "G"], ["D", "#", "E", "X"], ["S", "F", "H", "I"]]
    for row in layout:
        print("   ".join(c + arrows[policy[c]] if c in policy else c + " " for c in row))


print("Күй       UP     DOWN     LEFT    RIGHT   Ең жақсы")
for s in states:
    q = {a: q_value(s, a, V) for a in actions}
    best = max(q, key=q.get)
    print(f"{s:<4}" + "".join(f"{q[a]:>9.3f}" for a in actions) + f"   {best}")

policy = get_policy(V)
print()
print("СОҢҒЫ САЯСАТ")
for s in states:
    print(f"π({s}) = {policy[s]} {arrows[policy[s]]}")
print()
print_grid(policy)

Күй       UP     DOWN     LEFT    RIGHT   Ең жақсы
A       6.220    5.527    6.024    6.901   RIGHT
B       7.211    7.211    6.330    8.119   RIGHT
C       8.398    6.125    7.151    9.385   RIGHT
D       5.937    4.577    5.247    5.247   UP
E       6.228    3.084    5.680   -6.724   UP
S       5.011    4.345    4.493    3.971   UP
F       3.888    3.888    4.279    4.285   RIGHT
H       5.019    4.149    3.997    2.909   UP
I      -7.319    2.598    2.773    1.156   LEFT

СОҢҒЫ САЯСАТ
π(A) = RIGHT →
π(B) = RIGHT →
π(C) = RIGHT →
π(D) = UP ↑
π(E) = UP ↑
π(S) = UP ↑
π(F) = RIGHT →
π(H) = UP ↑
π(I) = LEFT ←

A→   B→   C→   G 
D↑   #    E↑   X 
S↑   F→   H↑   I←


## 7-тапсырма — Эксперименттер

Әр экспериментте бір параметр өзгереді, қалғандары базалық: γ = 0.9, p = 0.8, X = −10, қадам = −0.1.

In [ ]:
experiments = [
    ("1: γ 0.9 → 0.2",                {"gamma": 0.2}),
    ("2: негізгі бағыт 0.8 → 0.6",     {"p_main": 0.6}),
    ("3: X марапаты -10 → -30",        {"r_danger": -30}),
    ("4: қадам марапаты -0.1 → -1",    {"r_step": -1}),
]

base_policy = policy
for name, change in experiments:
    gamma_e = change.get("gamma", 0.9)
    params = {k: v for k, v in change.items() if k != "gamma"}
    V_e, n = value_iteration(gamma=gamma_e, verbose=False, **params)
    pol_e = get_policy(V_e, gamma_e, **params)
    changed = [f"{s}: {base_policy[s]} → {pol_e[s]}" for s in states if pol_e[s] != base_policy[s]]
    print(f"=== Эксперимент {name} ({n} итерация) ===")
    print("V:", {s: round(V_e[s], 3) for s in states})
    print("Өзгерген әрекеттер:", changed if changed else "жоқ")
    print_grid(pol_e)
    print()

=== Эксперимент 1: γ 0.9 → 0.2 (6 итерация) ===
V: {'A': 0.101, 'B': 1.254, 'C': 8.147, 'D': -0.087, 'E': 0.218, 'S': -0.119, 'F': -0.116, 'H': -0.07, 'I': -0.124}
Өзгерген әрекеттер: ['I: LEFT → DOWN']
A→   B→   C→   G 
D↑   #    E↑   X 
S↑   F→   H↑   I↓

=== Эксперимент 2: негізгі бағыт 0.8 → 0.6 (39 итерация) ===
V: {'A': 5.25, 'B': 6.733, 'C': 8.165, 'D': 4.273, 'E': 4.087, 'S': 3.261, 'F': 2.595, 'H': 2.837, 'I': 1.464}
Өзгерген әрекеттер: ['E: UP → LEFT', 'F: RIGHT → LEFT', 'I: LEFT → DOWN']
A→   B→   C→   G 
D↑   #    E←   X 
S↑   F←   H↑   I↓

=== Эксперимент 3: X марапаты -10 → -30 (35 итерация) ===
V: {'A': 6.721, 'B': 7.912, 'C': 9.149, 'D': 5.78, 'E': 3.843, 'S': 4.874, 'F': 4.158, 'H': 3.56, 'I': 1.156}
Өзгерген әрекеттер: ['F: RIGHT → LEFT', 'H: UP → LEFT', 'I: LEFT → DOWN']
A→   B→   C→   G 
D↑   #    E↑   X 
S↑   F←   H←   I↓

=== Эксперимент 4: қадам марапаты -0.1 → -1 (16 итерация) ===
V: {'A': 4.512, 'B': 6.749, 'C': 9.075, 'D': 2.742, 'E': 5.092, 'S': 1.192, 'F': 1